In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "Fraud-detection-ML-V2" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_PATH = RAW_DATA_DIR / "fraudTrain.csv"
TEST_PATH = RAW_DATA_DIR / "fraudTest.csv"

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Training dataset not found:\n{TRAIN_PATH}"
    )

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Test dataset not found:\n{TEST_PATH}"
    )

df = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

print("========== RAW DATA LOADED ==========")
print("Training shape:", df.shape)
print("Test shape:", df_test.shape)
print("Training file:", TRAIN_PATH.name)
print("Test file:", TEST_PATH.name)
print("=====================================")

========== RAW DATA LOADED ==========
Training shape: (1296675, 23)
Test shape: (555719, 23)
Training file: fraudTrain.csv
Test file: fraudTest.csv


In [2]:
required_columns = [
    "cc_num",
    "merchant",
    "category",
    "amt",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "trans_date_trans_time",
    "is_fraud"
]

missing_train = [
    column
    for column in required_columns
    if column not in df.columns
]

missing_test = [
    column
    for column in required_columns
    if column not in df_test.columns
]

if missing_train:
    raise ValueError(
        f"Missing columns in training data: {missing_train}"
    )

if missing_test:
    raise ValueError(
        f"Missing columns in test data: {missing_test}"
    )

print("========== COLUMN VALIDATION ==========")
print("All required training columns present:", len(missing_train) == 0)
print("All required test columns present:", len(missing_test) == 0)
print("=======================================")

========== COLUMN VALIDATION ==========
All required training columns present: True
All required test columns present: True


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

df_test.columns = (
    df_test.columns
    .str.strip()
    .str.lower()
)

print("========== COLUMN STANDARDIZATION ==========")
print("Training columns:")
print(df.columns.tolist())
print()
print("Test columns:")
print(df_test.columns.tolist())
print("============================================")

========== COLUMN STANDARDIZATION ==========
Training columns:
['unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']

Test columns:
['unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


In [4]:
numeric_columns = [
    "amt",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "is_fraud"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    df_test[column] = pd.to_numeric(
        df_test[column],
        errors="coerce"
    )

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"],
    errors="coerce"
)

df_test["trans_date_trans_time"] = pd.to_datetime(
    df_test["trans_date_trans_time"],
    errors="coerce"
)

print("========== TYPE CONVERSION ==========")
print(df.dtypes)
print("=====================================")

========== TYPE CONVERSION ==========
unnamed: 0                        int64
trans_date_trans_time    datetime64[ns]
cc_num                            int64
merchant                         object
category                         object
amt                             float64
first                            object
last                             object
gender                           object
street                           object
city                             object
state                            object
zip                               int64
lat                             float64
long                            float64
city_pop                          int64
job                              object
dob                              object
trans_num                        object
unix_time                         int64
merch_lat                       float64
merch_long                      float64
is_fraud                          int64
dtype: object


In [5]:
invalid_train = pd.DataFrame({
    "missing_values": df.isnull().sum(),
    "negative_amount": [
        int((df["amt"] < 0).sum())
        if column == "amt"
        else 0
        for column in df.columns
    ]
})

invalid_test = pd.DataFrame({
    "missing_values": df_test.isnull().sum()
})

print("========== INVALID VALUE CHECK ==========")
print("Training missing values:")
print(df.isnull().sum())

print()
print("Test missing values:")
print(df_test.isnull().sum())

print()
print("Training negative amounts:", int((df["amt"] < 0).sum()))
print("Test negative amounts:", int((df_test["amt"] < 0).sum()))

print("=========================================")

========== INVALID VALUE CHECK ==========
Training missing values:
unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64

Test missing values:
unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city   

In [6]:
before_train = len(df)
before_test = len(df_test)

essential_columns = [
    "cc_num",
    "merchant",
    "category",
    "amt",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "trans_date_trans_time",
    "is_fraud"
]

df = df.dropna(
    subset=essential_columns
)

df_test = df_test.dropna(
    subset=essential_columns
)

df = df[df["amt"] >= 0]
df_test = df_test[df_test["amt"] >= 0]

df = df[
    df["is_fraud"].isin([0, 1])
]

df_test = df_test[
    df_test["is_fraud"].isin([0, 1])
]

print("========== INVALID ROW REMOVAL ==========")
print("Training rows before:", before_train)
print("Training rows after:", len(df))
print("Training rows removed:", before_train - len(df))

print()

print("Test rows before:", before_test)
print("Test rows after:", len(df_test))
print("Test rows removed:", before_test - len(df_test))
print("=========================================")

========== INVALID ROW REMOVAL ==========
Training rows before: 1296675
Training rows after: 1296675
Training rows removed: 0

Test rows before: 555719
Test rows after: 555719
Test rows removed: 0


In [7]:
train_duplicates_before = int(
    df.duplicated().sum()
)

test_duplicates_before = int(
    df_test.duplicated().sum()
)

df = df.drop_duplicates().reset_index(
    drop=True
)

df_test = df_test.drop_duplicates().reset_index(
    drop=True
)

print("========== DUPLICATE REMOVAL ==========")
print(
    "Training duplicates removed:",
    train_duplicates_before
)

print(
    "Test duplicates removed:",
    test_duplicates_before
)

print(
    "Training shape:",
    df.shape
)

print(
    "Test shape:",
    df_test.shape
)

print("=======================================")

========== DUPLICATE REMOVAL ==========
Training duplicates removed: 0
Test duplicates removed: 0
Training shape: (1296675, 23)
Test shape: (555719, 23)


In [8]:
df = df.sort_values(
    ["cc_num", "trans_date_trans_time"]
).reset_index(
    drop=True
)

df_test = df_test.sort_values(
    ["cc_num", "trans_date_trans_time"]
).reset_index(
    drop=True
)

print("========== CHRONOLOGICAL SORT ==========")

print(
    "Training sorted:",
    df[
        ["cc_num", "trans_date_trans_time"]
    ].head()
)

print()

print(
    "Test sorted:",
    df_test[
        ["cc_num", "trans_date_trans_time"]
    ].head()
)

print("========================================")

========== CHRONOLOGICAL SORT ==========
Training sorted:         cc_num trans_date_trans_time
0  60416207185   2019-01-01 12:47:15
1  60416207185   2019-01-02 08:44:57
2  60416207185   2019-01-02 08:47:36
3  60416207185   2019-01-02 12:38:14
4  60416207185   2019-01-02 13:10:46

Test sorted:         cc_num trans_date_trans_time
0  60416207185   2020-06-21 13:05:42
1  60416207185   2020-06-21 16:25:36
2  60416207185   2020-06-22 07:58:33
3  60416207185   2020-06-22 15:32:31
4  60416207185   2020-06-23 12:28:54


In [9]:
train_time_order_valid = (
    df.groupby("cc_num")[
        "trans_date_trans_time"
    ]
    .apply(
        lambda x: x.is_monotonic_increasing
    )
    .all()
)

test_time_order_valid = (
    df_test.groupby("cc_num")[
        "trans_date_trans_time"
    ]
    .apply(
        lambda x: x.is_monotonic_increasing
    )
    .all()
)

print("========== CHRONOLOGY VALIDATION ==========")
print(
    "Training chronological order valid:",
    train_time_order_valid
)

print(
    "Test chronological order valid:",
    test_time_order_valid
)

print("===========================================")

if not train_time_order_valid:
    raise ValueError(
        "Training transactions are not correctly ordered."
    )

if not test_time_order_valid:
    raise ValueError(
        "Test transactions are not correctly ordered."
    )

========== CHRONOLOGY VALIDATION ==========
Training chronological order valid: True
Test chronological order valid: True


In [10]:
print("========== TARGET VALIDATION ==========")

print("Training target values:")
print(
    df["is_fraud"]
    .value_counts()
    .sort_index()
)

print()

print("Test target values:")
print(
    df_test["is_fraud"]
    .value_counts()
    .sort_index()
)

print()

print(
    "Training target valid:",
    set(df["is_fraud"].unique()).issubset({0, 1})
)

print(
    "Test target valid:",
    set(df_test["is_fraud"].unique()).issubset({0, 1})
)

print("======================================")

========== TARGET VALIDATION ==========
Training target values:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Test target values:
is_fraud
0    553574
1      2145
Name: count, dtype: int64

Training target valid: True
Test target valid: True


In [11]:
df.insert(
    0,
    "transaction_id",
    [
        f"TRN_{index:09d}"
        for index in range(1, len(df) + 1)
    ]
)

df_test.insert(
    0,
    "transaction_id",
    [
        f"TST_{index:09d}"
        for index in range(1, len(df_test) + 1)
    ]
)

print("========== TRANSACTION IDS ==========")
print(df["transaction_id"].head())
print()
print(df_test["transaction_id"].head())
print("=====================================")

========== TRANSACTION IDS ==========
0    TRN_000000001
1    TRN_000000002
2    TRN_000000003
3    TRN_000000004
4    TRN_000000005
Name: transaction_id, dtype: object

0    TST_000000001
1    TST_000000002
2    TST_000000003
3    TST_000000004
4    TST_000000005
Name: transaction_id, dtype: object


In [12]:
df["user_id"] = (
    df["cc_num"]
    .astype(str)
)

df_test["user_id"] = (
    df_test["cc_num"]
    .astype(str)
)

print("========== USER IDENTIFIER ==========")
print(
    "Unique training users:",
    df["user_id"].nunique()
)

print(
    "Unique test users:",
    df_test["user_id"].nunique()
)

print("=====================================")

========== USER IDENTIFIER ==========
Unique training users: 983
Unique test users: 924


In [13]:
cleaned_train_path = (
    PROCESSED_DATA_DIR
    / "cleaned_fraud_train.csv"
)

cleaned_test_path = (
    PROCESSED_DATA_DIR
    / "cleaned_fraud_test.csv"
)

df.to_csv(
    cleaned_train_path,
    index=False
)

df_test.to_csv(
    cleaned_test_path,
    index=False
)

print("========== CLEANED DATA SAVED ==========")
print(
    "Training output:",
    cleaned_train_path
)

print(
    "Test output:",
    cleaned_test_path
)

print(
    "Training file exists:",
    cleaned_train_path.exists()
)

print(
    "Test file exists:",
    cleaned_test_path.exists()
)

print("========================================")

========== CLEANED DATA SAVED ==========
Training output: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\data\processed\cleaned_fraud_train.csv
Test output: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\data\processed\cleaned_fraud_test.csv
Training file exists: True
Test file exists: True


In [14]:
print()
print("================================================")
print("       STREAMSENTINEL V2 — PHASE 2 SUMMARY")
print("================================================")

print()

print(
    "Training rows:",
    len(df)
)

print(
    "Test rows:",
    len(df_test)
)

print(
    "Training columns:",
    len(df.columns)
)

print(
    "Test columns:",
    len(df_test.columns)
)

print(
    "Training missing values:",
    int(df.isnull().sum().sum())
)

print(
    "Test missing values:",
    int(df_test.isnull().sum().sum())
)

print(
    "Training duplicates:",
    int(df.duplicated().sum())
)

print(
    "Test duplicates:",
    int(df_test.duplicated().sum())
)

print(
    "Training chronology valid:",
    train_time_order_valid
)

print(
    "Test chronology valid:",
    test_time_order_valid
)

print(
    "Training target valid:",
    set(df["is_fraud"].unique()).issubset({0, 1})
)

print(
    "Test target valid:",
    set(df_test["is_fraud"].unique()).issubset({0, 1})
)

print()
print("Phase 2 Data Preprocessing: COMPLETE")
print("================================================")


       STREAMSENTINEL V2 — PHASE 2 SUMMARY

Training rows: 1296675
Test rows: 555719
Training columns: 25
Test columns: 25
Training missing values: 0
Test missing values: 0
Training duplicates: 0
Test duplicates: 0
Training chronology valid: True
Test chronology valid: True
Training target valid: True
Test target valid: True

Phase 2 Data Preprocessing: COMPLETE
